# 13.6 — Speaker identification & diarization

Speaker systems compare voice embeddings, then diarization asks who spoke when across time. This gap topic is rebuilt as a real NumPy diarization pipeline: embeddings, cosine verification, k-means clustering, and overlap-aware evaluation on synthetic audio. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1305
rng = np.random.default_rng(SEED)
FS = 8000


def sine(freq, seconds=0.35, fs=FS, phase=0.0, amp=1.0):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    return amp * np.sin(2 * np.pi * freq * t + phase)


def mix_tones(freqs, seconds=0.35, fs=FS, amps=None):
    if amps is None:
        amps = np.ones(len(freqs))
    x = np.zeros(int(seconds * fs))
    for freq, amp in zip(freqs, amps):
        x = x + sine(freq, seconds, fs, amp=amp)
    scale = np.max(np.abs(x)) + 1e-12
    return x / scale


def chirp(start, stop, seconds=0.6, fs=FS):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    k = (stop - start) / max(seconds, 1e-12)
    phase = 2 * np.pi * (start * t + 0.5 * k * t * t)
    return np.sin(phase)


def add_noise(x, scale, local_rng=rng):
    return x + local_rng.normal(0.0, scale, size=len(x))


def stft_mag(x, n_fft=256, hop=96):
    if len(x) < n_fft:
        x = np.pad(x, (0, n_fft - len(x)))
    frames = []
    window = np.hanning(n_fft)
    for start in range(0, len(x) - n_fft + 1, hop):
        frame = x[start:start + n_fft]
        spec = np.abs(np.fft.rfft(frame * window))
        frames.append(spec)
    if not frames:
        frames.append(np.abs(np.fft.rfft(x[:n_fft] * window)))
    return np.array(frames)


def frame_audio(x, frame_size=256, hop=128):
    frames = []
    for start in range(0, len(x) - frame_size + 1, hop):
        frames.append(x[start:start + frame_size])
    if not frames:
        frames.append(np.pad(x, (0, max(0, frame_size - len(x))))[:frame_size])
    return np.array(frames)


def normalize_rows(values):
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-12)


def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / max(denom, 1e-12))


def softmax(logits):
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def top_frequency(x, fs=FS, n_fft=512):
    windowed = x[:n_fft]
    if len(windowed) < n_fft:
        windowed = np.pad(windowed, (0, n_fft - len(windowed)))
    mag = np.abs(np.fft.rfft(windowed * np.hanning(n_fft)))
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    return float(freqs[int(np.argmax(mag))])


def ladder_summary(rungs):
    for rung in rungs:
        x = rung["audio"]
        label = rung.get("label", rung.get("target", "synthetic"))
        print(rung["name"], x.shape, label)



def speaker_features(segment):
    spec = stft_mag(segment, n_fft=256, hop=96)
    log_spec = np.log1p(spec)
    freqs = np.arange(log_spec.shape[1])
    energy = np.sum(log_spec, axis=1) + 1e-12
    centroid = np.sum(log_spec * freqs, axis=1) / energy
    bandwidth = np.sqrt(np.sum(log_spec * (freqs[None, :] - centroid[:, None]) ** 2, axis=1) / energy)
    return np.array([
        np.mean(centroid),
        np.std(centroid),
        np.mean(bandwidth),
        np.std(bandwidth),
        np.log(np.mean(segment ** 2) + 1e-12),
    ])


def speaker_verify(query, enrollments, threshold=0.75):
    scores = {name: cosine(query, emb) for name, emb in enrollments.items()}
    best_name = max(scores, key=scores.get)
    accepted = scores[best_name] > threshold
    return best_name, scores, accepted


def kmeans_cosine(embeddings, k, steps=12):
    data = normalize_rows(embeddings)
    centers = data[:k].copy()
    for _ in range(steps):
        sims = data @ centers.T
        labels = np.argmax(sims, axis=1)
        for cluster in range(k):
            members = data[labels == cluster]
            if len(members) > 0:
                centers[cluster] = normalize_rows(np.mean(members, axis=0, keepdims=True))[0]
    return labels, centers


def best_cluster_accuracy(true_labels, pred_labels, k):
    best = 0
    for flip in range(k):
        mapped = (pred_labels + flip) % k
        best = max(best, int(np.sum(mapped == true_labels)))
    return best / len(true_labels)


def make_speaker_ladder():
    a = sine(220, seconds=0.30)
    b = sine(440, seconds=0.30)
    d1_audio = a
    d2_audio = np.concatenate([a, b, a, b])
    d3_audio = np.concatenate([add_noise(1.4 * a, 0.05), add_noise(0.6 * b, 0.05), add_noise(a, 0.05), add_noise(b, 0.05)])
    d4_audio = np.concatenate([chirp(210, 260, 0.25), chirp(420, 500, 0.25), sine(230, 0.25), sine(460, 0.25), mix_tones([220, 440], 0.25)])
    overlap = 0.7 * sine(220, 0.35) + 0.7 * sine(440, 0.35)
    d5_audio = np.concatenate([add_noise(sine(220, 0.35), 0.10), add_noise(overlap, 0.12), add_noise(1.8 * sine(440, 0.35), 0.10), add_noise(sine(220, 0.35), 0.12)])
    return [
        {"name": "D1 one speaker", "audio": d1_audio, "labels": np.array([0]), "k": 1},
        {"name": "D2 two speakers", "audio": d2_audio, "labels": np.array([0, 1, 0, 1]), "k": 2},
        {"name": "D3 noise and gain", "audio": d3_audio, "labels": np.array([0, 1, 0, 1]), "k": 2},
        {"name": "D4 conversation", "audio": d4_audio, "labels": np.array([0, 1, 0, 1, 2]), "k": 3},
        {"name": "D5 overlap", "audio": d5_audio, "labels": np.array([0, 2, 1, 0]), "k": 3},
    ]


def diarize_rung(rung):
    segments = np.array_split(rung["audio"], len(rung["labels"]))
    embeddings = np.array([speaker_features(segment) for segment in segments])
    labels, centers = kmeans_cosine(embeddings, rung["k"])
    purity = best_cluster_accuracy(rung["labels"], labels, rung["k"])
    return {"segments": segments, "embeddings": embeddings, "pred": labels, "purity": purity, "centers": centers}


## The concept, built once (D1): verification by cosine

Speaker comparison uses $\operatorname{cos}(e,a)=rac{e^	op a}{\|e\|_2\|a\|_2}$. The lesson query $e=[1,2]$ has $\cos(e,A)=0.998$ for $A=[1.2,2.1]$ and $\cos(e,B)=-0.447$ for $B=[-1,0]$.

In [ ]:

query = np.array([1.0, 2.0])
speaker_a = np.array([1.2, 2.1])
speaker_b = np.array([-1.0, 0.0])
name, scores, accepted = speaker_verify(query, {"A": speaker_a, "B": speaker_b}, threshold=0.75)
print(scores, name, accepted)
assert round(scores["A"], 3) == 0.998
assert round(scores["B"], 3) == -0.447
assert accepted is True


## Diarization adds clustering and overlap

The verification threshold $0.75$ accepts $0.998\gt0.75$ and rejects $-0.447\lt0.75$. For timeline reasoning, the lesson overlap example has one overlapped second in a four-second clip, so the overlap fraction is $1/4=0.250$.

In [ ]:

threshold = 0.75
accept_a = scores["A"] > threshold
reject_b = scores["B"] < threshold
overlap_fraction = 1.0 / 4.0
print("accept A", accept_a, "reject B", reject_b)
print("overlap fraction", round(overlap_fraction, 3))
assert accept_a is True
assert reject_b is True
assert round(overlap_fraction, 3) == 0.250


## The dataset ladder

F7 audio ladder inline: one sine speaker $ightarrow$ two-tone speakers $ightarrow$ noise/mic gain $ightarrow$ multi-segment conversation $ightarrow$ longer noisy overlap.

In [ ]:

rungs = make_speaker_ladder()
ladder_summary(rungs)
print("sample", np.round(rungs[0]["audio"][:8], 3))


In [ ]:

results = []
for rung in rungs:
    result = diarize_rung(rung)
    results.append(result)
    print(rung["name"], "purity", round(result["purity"], 3), "pred", result["pred"].tolist())
metrics = np.array([item["purity"] for item in results])


## Results visualization

The closing figure has one artifact panel per D1–D5 rung plus a metric curve over the same ladder.

In [ ]:

fig, axes = plt.subplots(2, len(rungs), figsize=(15, 5))
for idx, rung in enumerate(rungs):
    axes[0, idx].plot(rung["audio"], linewidth=0.6)
    axes[0, idx].set_title(rung["name"])
    pred = results[idx]["pred"]
    axes[1, idx].imshow(pred[None, :], aspect="auto", cmap="tab10", vmin=0, vmax=max(2, rung["k"]))
    axes[1, idx].set_yticks([])
    axes[1, idx].set_xlabel("segment")
fig.suptitle("Waveform and diarization timeline panels")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, 6), metrics, marker="o")
plt.xticks(np.arange(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(-0.05, 1.05)
plt.ylabel("clustering purity")
plt.title("Assignment accuracy vs complexity")
plt.show()


## Pitfall on D5: dot products confuse loudness with identity

Unnormalized dot products let gain masquerade as speaker identity. The fix is cosine normalization plus an overlap-aware label for mixed segments.

In [ ]:

d5 = rungs[-1]
segments = np.array_split(d5["audio"], len(d5["labels"]))
embeddings = np.array([speaker_features(segment) for segment in segments])
dot_scores = embeddings @ embeddings[0]
cos_scores = normalize_rows(embeddings) @ normalize_rows(embeddings[[0]])[0]
print("dot scores", np.round(dot_scores, 2))
print("cos scores", np.round(cos_scores, 3))
print("overlap-aware labels", d5["labels"].tolist())


## Evaluate it + Practice

- Metric: segment assignment accuracy or clustering purity; no-skill baseline assigns the largest speaker cluster.
- Sanity check: same-speaker cosine should beat impostor cosine on D1/D2.
- Ablation: replace cosine with dot products and watch gain dominate D3/D5.
- Failure signals: empty clusters, label switching without purity matching, and all overlap forced into one speaker.

Practice 1: increase D3 microphone gain and compare dot-score versus cosine behavior.


Practice 2: add a third non-overlapped speaker segment and rerun k-means with $k=3$.